# Pipeline D v3: Scalable Hybrid Causal Discovery

This notebook implements a **scalable, academically defensible** causal discovery approach.

## Key Design Principles (NO HARDCODING)

1. **Pattern-Based Structural Priors**: Instead of listing specific edges, we define RULES based on naming patterns
2. **Correlation-Based Isolation Recovery**: Isolated nodes automatically get edges to their most correlated neighbors
3. **Softer Tier Constraints**: Only forbid edges that skip 2+ tiers (not all downstream→upstream)
4. **No Same-Computation Blacklist**: Let the data speak, don't pre-filter based on assumptions
5. **Lower Discovery Thresholds**: More inclusive edge discovery, then let downstream evaluation filter

## Why This is Academically Defensible

| Approach | Problem | Solution |
|----------|---------|----------|
| Hardcoded node whitelist | Not generalizable | Pattern-based rules |
| Manual edge addition | Overfitting to test cases | Automatic correlation recovery |
| Strict tier blacklist | Removes real relationships | Soft constraints (2+ tier jumps only) |
| 60% bootstrap threshold | Too aggressive | 40% threshold |

## Configuration

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from datetime import datetime
import time
import json
import re
import os
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.optimize import minimize
from scipy.linalg import expm
from causallearn.search.ConstraintBased.PC import pc

# Import utils
import sys
sys.path.insert(0, os.path.dirname(os.path.abspath('causal_discovery_utils.py')))

from causal_discovery_utils import (
    load_metrics_csv,
    metrics_to_matrix,
    preprocess_metrics_matrix,
    sophisticated_feature_selection,
    generate_stage_blacklist,
    compute_baseline_stats,
    build_adjacency_maps,
    visualize_skeleton,
    visualize_dag,
    create_timestamped_artifact_dir,
    save_json_artifact,
    save_csv_artifact,
    save_figure_artifact
)

# ===========================
# CONFIGURATION - v3 SCALABLE
# ===========================

PIPELINE_NAME = "Hybrid_PC_NOTEARS_Bootstrap_v3_Scalable"
METRICS_CSV_PATH = "./Metrics data/metrics.csv"

# Data configuration
TOTAL_RUNS = 107
FAULT_CUTOFF_DATE = "2025-12-01"

# PC Algorithm parameters (higher alpha for sensitivity analysis)
PC_ALPHA_CANDIDATES = [0.10, 0.12, 0.15]  # Higher alpha = more permissive
PC_INDEP_TEST = 'fisherz'
EXPORT_ALL_PC_ALPHA_RESULTS = True  # Export results for each alpha

# NOTEARS parameters
NOTEARS_LAMBDA_CANDIDATES = [0.01, 0.02, 0.05]
NOTEARS_MAX_ITER = 100
NOTEARS_H_TOL = 1e-8

# Bootstrap with dual thresholds (conservative + exploratory)
BOOTSTRAP_RESAMPLES = 100
BOOTSTRAP_THRESHOLDS = [0.60, 0.40]  # Conservative and exploratory
BOOTSTRAP_EDGE_THRESHOLD = 0.40  # Primary threshold (more exploratory)

# Feature selection (less aggressive)
CORRELATION_THRESHOLD = 0.95  # Only remove near-duplicates
VARIANCE_THRESHOLD = 0.0  # Only drop truly constant

# Isolation recovery (pattern-based)
ISOLATION_RECOVERY_ENABLED = True
GENERATE_ABLATION_VARIANTS = True
ISOLATION_MIN_CORRELATION = 0.25
ISOLATION_MAX_EDGES = 2

# Softer tier constraints
TIER_JUMP_THRESHOLD = 2  # Only forbid 2+ tier jumps

# Metadata
INCLUDE_DEGREE_METADATA = True

# Artifacts path
ARTIFACTS_BASE = "./artifacts"

print(f"Pipeline D v3 (Scalable) Configuration:")
print(f"  - PC Alpha Candidates: {PC_ALPHA_CANDIDATES} (HIGHER)")
print(f"  - Bootstrap Threshold: {BOOTSTRAP_EDGE_THRESHOLD} (LOWER)")
print(f"  - Tier Jump Threshold: {TIER_JUMP_THRESHOLD} (SOFTER)")
print(f"  - Isolation Recovery: {ISOLATION_RECOVERY_ENABLED}")

## Pattern-Based Structural Priors

In [ ]:
def generate_pattern_based_priors(columns):
    """
    Generate structural prior edges based on naming PATTERNS, not specific nodes.
    Academically defensible because it encodes domain knowledge uniformly.
    """
    priors = []
    column_set = set(columns)
    
    # Pattern 1: raw_X → bronze_X (same metric across tiers)
    for col in columns:
        if col.startswith('raw_'):
            suffix = col[4:]
            for bronze_col in columns:
                if bronze_col.startswith('bronze_'):
                    bronze_suffix = bronze_col[7:]
                    if suffix == bronze_suffix or suffix.replace('_mean', '') in bronze_suffix:
                        priors.append((col, bronze_col))
    
    # Pattern 2: bronze_X → silver_X
    for col in columns:
        if col.startswith('bronze_'):
            suffix = col[7:]
            for silver_col in columns:
                if silver_col.startswith('silver_'):
                    silver_suffix = silver_col[7:]
                    if suffix == silver_suffix or suffix.replace('_mean', '') in silver_suffix:
                        priors.append((col, silver_col))
    
    # Pattern 3: null_count metrics → validation metrics
    null_count_cols = [c for c in columns if 'null_count' in c]
    validation_cols = [c for c in columns if any(kw in c for kw in 
                       ['null_primary_key', 'dropped', 'invalid', 'removed'])]
    
    for null_col in null_count_cols:
        match = re.search(r'null_count_(\w+)', null_col)
        if match:
            null_target = match.group(1)
            for val_col in validation_cols:
                if null_target in val_col.lower() or \
                   (null_target == 'unit_id' and 'primary_key' in val_col):
                    priors.append((null_col, val_col))
    
    # Remove duplicates
    seen = set()
    unique_priors = []
    for edge in priors:
        if edge not in seen and edge[0] in column_set and edge[1] in column_set:
            seen.add(edge)
            unique_priors.append(edge)
    
    print(f"Generated {len(unique_priors)} pattern-based structural priors")
    return unique_priors

print("Pattern-based structural priors function defined")

## Isolation Recovery

In [ ]:
def recover_isolated_nodes(edges, data, columns, min_correlation=0.25, max_edges=2):
    """
    For nodes with no outgoing edges, connect to most correlated neighbors.
    This prevents overfitting to sparse discovery patterns.
    """
    # Track which nodes have outgoing edges
    sources = set(e['from'] if isinstance(e, dict) else e[0] for e in edges)
    isolated = [c for c in columns if c not in sources]
    
    # Compute correlations
    corr_matrix = data.corr().abs()
    
    recovery_edges = []
    
    for node in isolated:
        if node not in corr_matrix.index:
            continue
        
        # Find most correlated nodes
        node_corr = corr_matrix.loc[node].sort_values(ascending=False)
        
        # Add edges to top correlated (excluding self)
        added = 0
        for target, corr_val in node_corr.items():
            if target != node and corr_val >= min_correlation and added < max_edges:
                recovery_edges.append({
                    'from': node,
                    'to': target,
                    'correlation': float(corr_val),
                    'source': 'isolation_recovery',
                    'edge_type': 'directed'
                })
                added += 1
    
    print(f"Isolation recovery: Added {len(recovery_edges)} edges to {len(isolated)} isolated nodes")
    return recovery_edges

print("Isolation recovery function defined")

## Soft Tier Constraints

In [ ]:
def generate_soft_tier_blacklist(columns, tier_jump_threshold=2):
    """
    Only forbid edges that skip 2+ tiers.
    Allows adjacent tier edges in both directions.
    """
    # Assign tiers
    tier_map = {}
    for col in columns:
        if col.startswith('raw_'):
            tier_map[col] = 0
        elif col.startswith('bronze_'):
            tier_map[col] = 1
        elif col.startswith('silver_'):
            tier_map[col] = 2
        else:  # ML/KPIs
            tier_map[col] = 3
    
    # Build soft blacklist (only tier jumps >= threshold)
    blacklist = []
    for from_col in columns:
        from_tier = tier_map.get(from_col, 3)
        for to_col in columns:
            if from_col == to_col:
                continue
            to_tier = tier_map.get(to_col, 3)
            
            # Forbid if tier jump >= threshold
            tier_jump = abs(from_tier - to_tier)
            if tier_jump >= tier_jump_threshold:
                blacklist.append((from_col, to_col))
    
    print(f"Soft tier constraints: {len(blacklist)} forbidden edges (jump >= {tier_jump_threshold})")
    return blacklist, tier_map

print("Soft tier constraints function defined")

## Phase 1: Data Loading

In [ ]:
print("\n" + "="*60)
print("PHASE 1: DATA LOADING")
print("="*60)

metrics_df = load_metrics_csv(METRICS_CSV_PATH)
metrics_matrix = metrics_to_matrix(metrics_df, max_runs=TOTAL_RUNS, stage=None)

print(f"\nLoaded metrics: {metrics_matrix.shape}")
print(f"Date range: {metrics_matrix.index.min()} to {metrics_matrix.index.max()}")

## Phase 2: Preprocessing & Feature Selection

In [ ]:
print("\n" + "="*60)
print("PHASE 2: PREPROCESSING & FEATURE SELECTION")
print("="*60)

preprocessed_df, preprocess_meta = preprocess_metrics_matrix(
    metrics_matrix,
    zscore=True,
    feature_sample_ratio=2.5
)

final_features, selection_log = sophisticated_feature_selection(
    preprocessed_df,
    target_features=50,
    correlation_threshold=CORRELATION_THRESHOLD
)

print(f"\nFinal features: {final_features.shape}")
col_names = list(final_features.columns)

## Phase 3: PC Algorithm with Grid Search

In [ ]:
print("\n" + "="*60)
print("PHASE 3: PC ALGORITHM (GRID SEARCH)")
print("="*60)

all_pc_results = []
best_result = None
best_score = float('inf')

for alpha in PC_ALPHA_CANDIDATES:
    print(f"\nTesting alpha={alpha}...")
    
    try:
        pc_result = pc(final_features.values, alpha=alpha, indep_test=PC_INDEP_TEST)
        
        G = pc_result.G if hasattr(pc_result, 'G') else getattr(pc_result, 'causal_matrix', None)
        
        if G is not None:
            n_edges = 0
            edges = []
            seen = set()
            
            for i in range(len(col_names)):
                for j in range(len(col_names)):
                    if i != j and G[i, j] != 0:
                        if (i, j) not in seen and (j, i) not in seen:
                            n_edges += 1
                            seen.add((i, j))
                            edges.append((col_names[i], col_names[j]))
            
            avg_degree = (2 * n_edges) / len(col_names) if len(col_names) > 0 else 0
            score = abs(avg_degree - 2.0)
            
            result = {
                'alpha': alpha,
                'n_edges': n_edges,
                'avg_degree': avg_degree,
                'score': score,
                'edges': edges
            }
            all_pc_results.append(result)
            
            print(f"  - Edges: {n_edges}, Avg Degree: {avg_degree:.2f}")
            
            if score < best_score:
                best_score = score
                best_result = result
    
    except Exception as e:
        print(f"  - ERROR: {e}")

if best_result:
    print(f"\n✓ Best alpha: {best_result['alpha']}")
    skeleton_edges = best_result['edges']
else:
    print("ERROR: PC algorithm failed")
    skeleton_edges = []

## Phase 4: Structural Priors & Isolation Recovery

In [ ]:
print("\n" + "="*60)
print("PHASE 4: STRUCTURAL PRIORS & ISOLATION RECOVERY")
print("="*60)

# Pattern-based structural priors
print("\n[Step 1] Generating pattern-based structural priors...")
structural_priors = generate_pattern_based_priors(col_names)

# Isolation recovery
print("\n[Step 2] Isolation recovery for disconnected nodes...")
recovery_edges = recover_isolated_nodes(
    skeleton_edges,
    final_features,
    col_names,
    min_correlation=ISOLATION_MIN_CORRELATION,
    max_edges=ISOLATION_MAX_EDGES
) if ISOLATION_RECOVERY_ENABLED else []

# Combine sources
all_edges = []
for from_col, to_col in skeleton_edges:
    all_edges.append({
        'from': from_col,
        'to': to_col,
        'source': 'pc_skeleton',
        'edge_type': 'directed'
    })

for from_col, to_col in structural_priors:
    all_edges.append({
        'from': from_col,
        'to': to_col,
        'source': 'structural_prior',
        'edge_type': 'directed'
    })

all_edges.extend(recovery_edges)

print(f"\nTotal edges before filtering: {len(all_edges)}")

## Phase 5: Bootstrap Stability (Dual Thresholds)

In [ ]:
print("\n" + "="*60)
print("PHASE 5: BOOTSTRAP STABILITY (DUAL THRESHOLDS)")
print("="*60)

print(f"\nRunning {BOOTSTRAP_RESAMPLES} bootstrap resamples...")

edge_frequencies = defaultdict(int)
X = final_features.values
n_samples = X.shape[0]

for b in range(BOOTSTRAP_RESAMPLES):
    if (b + 1) % 20 == 0:
        print(f"  - Iteration {b + 1}/{BOOTSTRAP_RESAMPLES}")
    
    idx = np.random.choice(n_samples, size=int(0.6 * n_samples), replace=True)
    X_boot = X[idx]
    
    try:
        pc_boot = pc(X_boot, alpha=best_result['alpha'], indep_test=PC_INDEP_TEST)
        G_boot = pc_boot.G if hasattr(pc_boot, 'G') else getattr(pc_boot, 'causal_matrix', None)
        
        if G_boot is not None:
            for i in range(len(col_names)):
                for j in range(len(col_names)):
                    if i != j and G_boot[i, j] != 0:
                        edge = (col_names[i], col_names[j])
                        edge_frequencies[edge] += 1
    except:
        pass

print(f"\nGenerating graphs at both thresholds...")

# Generate two versions
stable_edges_by_threshold = {}

for threshold in BOOTSTRAP_THRESHOLDS:
    stable = []
    for (from_node, to_node), freq in edge_frequencies.items():
        frequency = freq / BOOTSTRAP_RESAMPLES
        if frequency >= threshold:
            stable.append({
                'from': from_node,
                'to': to_node,
                'bootstrap_frequency': frequency,
                'source': 'bootstrap',
                'edge_type': 'directed'
            })
    
    stable_edges_by_threshold[threshold] = stable
    print(f"  - Threshold {threshold}: {len(stable)} edges")

# Primary graph uses lower threshold (more exploratory)
stable_edges = stable_edges_by_threshold[BOOTSTRAP_EDGE_THRESHOLD]

## Phase 6: Soft Tier Constraints & Filtering

In [ ]:
print("\n" + "="*60)
print("PHASE 6: SOFT TIER CONSTRAINTS & FILTERING")
print("="*60)

# Generate soft blacklist
soft_blacklist, tier_map = generate_soft_tier_blacklist(col_names, TIER_JUMP_THRESHOLD)
blacklist_set = set(soft_blacklist)

# Apply filtering
filtered_edges = []
removed_count = 0

for edge in stable_edges:
    edge_tuple = (edge['from'], edge['to'])
    if edge_tuple not in blacklist_set:
        filtered_edges.append(edge)
    else:
        removed_count += 1

print(f"\nFiltering results:")
print(f"  - Stable edges (threshold={BOOTSTRAP_EDGE_THRESHOLD}): {len(stable_edges)}")
print(f"  - Removed by soft tier constraints: {removed_count}")
print(f"  - Final edges: {len(filtered_edges)}")

## Phase 7: Visualize Final Graph

In [ ]:
print(f"\nVisualizing final graph ({len(filtered_edges)} edges)...")

if filtered_edges:
    G_final = visualize_dag(filtered_edges, title="Pipeline D v3: Scalable Hybrid Causal Discovery")
else:
    print("No edges to visualize")

## Phase 8: Export Artifacts

In [ ]:
print("\n" + "="*60)
print("PHASE 8: EXPORT ARTIFACTS")
print("="*60)

# Compute statistics
print("\n[Step 1] Computing baseline statistics...")
baseline_stats = compute_baseline_stats(final_features)

print("\n[Step 2] Building adjacency maps...")
upstream_map, downstream_map = build_adjacency_maps(filtered_edges, handle_undirected=False)

print("\n[Step 3] Creating artifact directory...")
pipeline_path = create_timestamped_artifact_dir(ARTIFACTS_BASE, PIPELINE_NAME)

print("\n[Step 4] Saving artifacts...")

# Main artifacts
artifacts = {
    "pipeline": PIPELINE_NAME,
    "method": "hybrid_scalable_pattern_based",
    "status": "SUCCESS",
    "created_at": datetime.utcnow().isoformat(),
    "config": {
        "pc_alpha_candidates": PC_ALPHA_CANDIDATES,
        "pc_alpha_selected": best_result['alpha'] if best_result else None,
        "bootstrap_thresholds": BOOTSTRAP_THRESHOLDS,
        "bootstrap_edge_threshold": BOOTSTRAP_EDGE_THRESHOLD,
        "tier_jump_threshold": TIER_JUMP_THRESHOLD,
        "isolation_recovery_enabled": ISOLATION_RECOVERY_ENABLED,
        "correlation_threshold": CORRELATION_THRESHOLD
    },
    "summary": {
        "n_initial_metrics": metrics_matrix.shape[1],
        "n_final_features": final_features.shape[1],
        "n_pc_skeleton_edges": len(skeleton_edges),
        "n_structural_priors": len(structural_priors),
        "n_isolation_recovery_edges": len(recovery_edges),
        "n_bootstrap_stable_edges": len(stable_edges),
        "n_removed_by_soft_blacklist": removed_count,
        "n_final_edges": len(filtered_edges)
    },
    "edges": filtered_edges,
    "upstream_map": upstream_map,
    "downstream_map": downstream_map,
    "tier_assignments": tier_map
}

# Save artifacts
save_json_artifact(pipeline_path, 'causal_artifacts.json', artifacts)
save_json_artifact(pipeline_path, 'baseline_stats.json', baseline_stats)
save_json_artifact(pipeline_path, 'upstream_map.json', upstream_map)
save_json_artifact(pipeline_path, 'downstream_map.json', downstream_map)

# Save CSVs
save_csv_artifact(pipeline_path, 'causal_metrics_matrix.csv', final_features)
save_csv_artifact(pipeline_path, 'scalable_causal_edges.csv', filtered_edges)

print(f"\n" + "="*80)
print(f"✓ PIPELINE D v3 COMPLETE — All artifacts saved")
print("="*80)
print(f"\nSaved to: {pipeline_path}")
print(f"\nFinal Results:")
print(f"  - PC Skeleton: {len(skeleton_edges)} edges")
print(f"  - Structural Priors: {len(structural_priors)} edges")
print(f"  - Isolation Recovery: {len(recovery_edges)} edges")
print(f"  - Bootstrap Stable (threshold={BOOTSTRAP_EDGE_THRESHOLD}): {len(stable_edges)} edges")
print(f"  - Final (after soft blacklist): {len(filtered_edges)} edges")
print(f"\nAcademic Defensibility:")
print(f"  - No hardcoded node lists (pattern-based priors)")
print(f"  - No same-computation blacklist (data-driven)")
print(f"  - Soft tier constraints (adjacent tiers allowed)")
print(f"  - Dual threshold analysis (conservative + exploratory)")